# Local LLM Bench — Results Analysis

מטלה 5, קוד קבוצה `bb-ai-12`. Notebook זה קורא תוצאות שמורות תחת `results/` (ראיות
אמיתיות שנאספו ב-Phase 4: baseline רשמי, AirLLM, קוונטיזציה Q4_0/Q2_K, וניתוח כלכלי)
ומציג ניתוח משווה בין שלוש השיטות.

**סטטוס:** הניסויים האמיתיים בוצעו (ר' `README.md` "יומן ניסויים" לניתוח המלא
והמנומק). Notebook זה משכפל את הטבלאות/הגרפים המרכזיים לצורך חקירה אינטראקטיבית —
המקור הקנוני לניתוח הוא README.md עצמו (`ex05` §8 דורש ש-README ישמש כדוח הסופי).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from local_llm_bench.sdk import LocalLLMBenchSDK

sdk = LocalLLMBenchSDK(project_root=Path.cwd().parent)
sdk.probe_hardware()

## הרצת מטריצת הניסויים המלאה (הוחלט שלא להריץ במלואה — ר' נימוק)

**החלטת היקף מתועדת**: `run_full_benchmark_suite()` מריץ קומבינציה מלאה של
2 פרומפטים × 3 אורכי-פלט × 3 שיטות = 18 ריצות. בהינתן ש-AirLLM נמדד בפועל ב-
~43.1 שניות/טוקן (ר' README), ריצה מלאה על אורך הפלט המקסימלי (128 טוקנים) הייתה
לוקחת לבדה מעל שעה לכל פרומפט — בסה"כ ריצה מלאה הייתה נמשכת מעל 5 שעות, בניגוד
מפורש להנחיית `ex05` §6 ("אל תהפכו את המטלה לפרויקט גמר"; §1 "שיקול חשוב").

**מה נעשה בפועל במקום זאת**: ריצות ממוקדות בודדות (באמצעות `sdk.run_baseline`/
`run_airllm`/`run_quantized` ישירות, לא דרך `run_full_benchmark_suite`) שכל אחת
מהן תועדה בנפרד תחת `results/` — כולל בדיקת רגישות מצומצמת (Q4_0 בשני אורכי
פלט שונים) במקום הרשת המלאה. ר' README "יומן ניסויים" לפירוט כל ריצה.

In [ ]:
# results_path = sdk.run_full_benchmark_suite()
# print(results_path)

## הפקת גרפים וטבלת השוואה מתוך תוצאות קיימות

In [ ]:
import pandas as pd

assets_path = sdk.generate_report(Path.cwd().parent / "results")
summary = pd.read_csv(assets_path / "summary_table.csv")
summary

## ניתוח רגישות: תפוקה (tokens/sec) מול אורך הפלט המבוקש

In [ ]:
from IPython.display import Image

Image(filename=str(assets_path / "tokens_per_sec_vs_length.png"))

## Peak RAM: Baseline מול AirLLM מול קוונטיזציה

In [ ]:
Image(filename=str(assets_path / "peak_ram_comparison.png"))

## מסקנות

**Baseline (טעינה ישירה, ~14B, FP32) נכשל תמיד** — דרך Ollama עם מודל 72B ("unable
to allocate CPU buffer" תוך 6.28s) ודרך `ModelLoaderService` הרשמי עם Phi-3-medium
עצמו (קריסת segfault תוך ~17.3s, ר' `results/baseline_official_phi3_medium_oom_crash.json`).
שני הכשלים קורים **לפני** כל חישוב Prefill/Decode ממשי — הוכחה ישירה
שהמגבלה היא **memory-bound**, לא compute-bound (עונה על שאלת מחקר #1).

**AirLLM מצליח באותה חומרה בדיוק** (903MB peak RAM בלבד) בזכות טעינה
שכבה-אחר-שכבה מהדיסק (mmap) — כל שכבה נטענת, מופעלת, ומשוחררת לפני הבאה, בדיוק
כמו Paging במערכות הפעלה (שאלת מחקר #2). המחיר: 43.1 שניות/טוקן, כי **כל** טוקן
דורש מעבר מחדש על כל 40 השכבות מהדיסק — אין "שמירה" בין טוקנים (שאלת מחקר #5).

**קוונטיזציה (Q4_0/Q2_K) יושבת בתווך**: מקטינה את המודל מ-28GB ל-~4-8GB (תלוי
ברמה) ומאיצה משמעותית מול AirLLM, אך עדיין טוענת את כל המודל המכווץ לזיכרון בבת
אחת (peak RAM גבוה יותר מ-AirLLM, ~4GB) — טרייד-אוף שונה לחלוטין: פחות latency,
יותר RAM (שאלת מחקר #3).

**TTFT מול TPOT (שאלת מחקר #4)**: בכל שיטה, TTFT (Prefill — בניית ה-KV cache על
כל טוקני הפרומפט בבת אחת) שונה מהותית מ-TPOT (Decode — טוקן בודד בכל פעם). ב-
AirLLM הפער דרמטי במיוחד: TTFT=79.4s (מעבר Prefill אחד על 40 השכבות) לעומת
TPOT=43.1s/token (כל טוקן Decode דורש מעבר חוזר נפרד) — מראה בבירור ששני
השלבים נמדדים ומתנהגים אחרת.

**כלכלית (שאלת מחקר #6)**: נקודת האיזון בין On-Prem ל-API היא ~10,000 בקשות/חודש
(ר' `assets/breakeven_analysis.png`) — מתחת לכך API עדיף (ללא השקעת CAPEX), מעליה
On-Prem זול משמעותית (החשמל זניח ביחס לחומרה המופחתת).

**Model Roofline**: כל הנקודות הנמדדות (airllm/quantized) נופלות משמעותית מתחת
לתקרת ה-roofline המשוערת — עדות נוספת שכל השיטות בהרכב החומרה הזה הן
memory/IO-bound ולא מנצלות את מלוא כוח החישוב התיאורטי, כי צוואר הבקבוק האמיתי
הוא זרימת הנתונים מהדיסק/RAM, לא ה-FLOPs עצמם.

לניתוח המלא, המנומק, וההשוואות המלאות — ר' `README.md` "יומן ניסויים".

In [ ]:
Image(filename=str(assets_path / "breakeven_analysis.png"))

## Model Roofline (ההרחבה המקורית — PLAN.md ADR-6)

ממחיש האם כל שיטה פועלת במשטר memory-bound (מתחת לתקרת רוחב הפס, בשיפוע העולה)
או מתקרבת לתקרת ה-compute (אזור אופקי). תקרות רוחב-הפס/GFLOPs הן **הנחות מוגדרות
בקונפיגורציה** (`config/setup.json: roofline`), לא נמדדו ישירות מהיצרן — מתועד
במפורש כך שהניתוח ניתן לשחזור ולעדכון.

In [ ]:
roofline_path = sdk.generate_model_roofline(Path.cwd().parent / "results", model_params_billion=14.0)
Image(filename=str(roofline_path))